# 03 - Quantum-Classical Hybrid (Track C)

**Purpose.** Attach a trainable variational quantum circuit (4 qubits, angle encoding, Rot layers, cyclic CNOT, Pauli-Z readout) to the best classical backbone (selected on validation) and compare it with parameter-matched classical controls under one predetermined budget, over several seeds. Record simulator cost and trainability diagnostics.

**Requires (inputs).** Notebook 00 output (manifest) and notebook 01/02 outputs (trained backbone checkpoint), same MODE.

**Produces (outputs, under `/kaggle/working/rsna_ich_artifacts/<MODE>/`).** `features/<backbone>/features_<split>.npz`, `experiments/C_<head>_s<seed>/`, `analysis/trackC_*.csv`, `analysis/vqc_circuit.txt`, `analysis/simulator_benchmark.csv`.

> Research software. Outputs are model predictions for a research study and **not** clinical diagnoses.
> Results in this notebook are only valid for the MODE they were produced in (`smoke` results check correctness and say nothing about performance).

In [ ]:
# ==== Setup: locate the project code and make `src` importable ====================================
# Works on Kaggle (project code attached as a Dataset, possibly nested after zip extraction) and locally.
import os, sys, importlib
from pathlib import Path

def find_project_root(max_depth: int = 5) -> Path:
    def ok(p: Path) -> bool:
        return (p / "src" / "__init__.py").is_file() and (p / "configs" / "data_config.yaml").is_file()
    cands = []
    if os.environ.get("RSNA_ICH_PROJECT_ROOT"):
        cands.append(Path(os.environ["RSNA_ICH_PROJECT_ROOT"]))
    cands += [Path.cwd(), Path.cwd().parent, Path("/kaggle/working/RSNA_ICH_Research")]
    for c in cands:
        if ok(c):
            return c.resolve()
    for base in (Path("/kaggle/input"), Path("/kaggle/working")):
        if not base.exists():
            continue
        stack = [(base, 0)]
        while stack:
            d, depth = stack.pop()
            if ok(d):
                return d.resolve()
            if depth < max_depth:
                try:
                    stack += [(p, depth + 1) for p in d.iterdir() if p.is_dir() and not p.name.startswith("stage_2_")]
                except OSError:
                    pass
    raise FileNotFoundError(
        "Project code not found. On Kaggle: 'Add Input' -> the dataset containing RSNA_ICH_Research "
        "(with src/ and configs/). Locally: run the notebook from RSNA_ICH_Research/notebooks or set "
        "the RSNA_ICH_PROJECT_ROOT environment variable.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
for _m in ["src", "src.utils.context", "src.data", "src.models", "src.quantum", "src.training", "src.evaluation"]:
    importlib.import_module(_m)  # fails loudly here if a dependency is missing
import src
print(f"Project root: {PROJECT_ROOT}  |  src version {src.__version__}")

In [ ]:
# ==== Optional dependencies ======================================================================
# Kaggle images ship torch, timm, pydicom, scikit-learn and OpenCV. PennyLane is only needed for the
# reference quantum backend (the default torch simulator does not need it). With internet enabled:
#   !pip install -q pennylane
# Without internet, attach a dataset of wheels and run: !pip install --no-index --find-links /kaggle/input/<wheels> pennylane
from src.utils.runtime import package_versions
print(package_versions())

In [ ]:
# ==== Configuration ================================================================================
# MODE: "smoke" (correctness check, minutes) | "pilot" (representative subset) | "full" (largest feasible).
MODE = "smoke"
# Optional per-config overrides without editing YAML, e.g. {"transformer": {"train": {"epochs": 2}}}
OVERRIDES = {}
import json
if os.environ.get("RSNA_ICH_OVERRIDES_JSON"):          # test hook used by tools/run_notebooks_synthetic.py
    OVERRIDES = json.loads(os.environ["RSNA_ICH_OVERRIDES_JSON"])
MODE = os.environ.get("RSNA_ICH_MODE", MODE)

from src.utils.context import ProjectContext
ctx = ProjectContext(PROJECT_ROOT, MODE, OVERRIDES)
print(ctx.describe())
print("Installed packages:", ctx.runtime.packages)
ctx.save_run_info("03_quantum")

In [ ]:
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from IPython.display import display
from src import LABELS
from src.utils.io import atomic_write_json, atomic_save_dataframe, read_json, dataframe_to_markdown
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

## 1. Manifest, configuration and backbone selection (validation only)

In [ ]:
from src.training.experiment import select_best_by_validation, load_model_from_result, split_frames, extract_features, run_head_experiment
manifest, man_info = ctx.manifest()
ctx.restore_experiments(("C_",))
qcfg = ctx.cfg["quantum"]
src_cfg = qcfg["backbone"]["source"]
if src_cfg == "auto":
    bb = select_best_by_validation(ctx, tuple(qcfg["backbone"]["candidates_prefix"]))
else:
    r = ctx.results((src_cfg,)); bb = r.iloc[0].to_dict() if len(r) else None
if bb is None:
    raise RuntimeError("No trained backbone found. Run/attach notebook 01 (and/or 02) for this MODE first.")
print("Backbone:", bb["experiment_id"], "| val macro AUC", round(bb["val_metrics"]["macro_roc_auc"], 4),
      "| weights", bb["build"]["status"])

## 2. Extract backbone features once (frozen backbone; restartable cache)
The same per-split frames as the backbone's own evaluation are used, so all heads see identical images.

In [ ]:
from src.data.preprocessing import PreprocessConfig
backbone = load_model_from_result(bb, man_info["manifest_hash"])
frames = split_frames(manifest, bb["config"]["limits"], bb["config"]["seed"])
pp = PreprocessConfig.from_dict(bb["config"]["preprocess"])
feat_dir = ctx.dir("features", bb["experiment_id"])
source_fp = f'{bb["experiment_id"]}:{bb["config_fingerprint"]}'
feats = extract_features(ctx, backbone, frames, pp, feat_dir,
                         {"source_fingerprint": source_fp, "manifest_hash": man_info["manifest_hash"]},
                         batch_size=qcfg["backbone"]["feature_batch_size"])
print({s: v["features"].shape for s, v in feats.items()})
del backbone; torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 3. The circuit, its resources and a backend cross-check
Two exact simulators are available: a batched PyTorch state-vector simulator (default) and PennyLane. They must agree; any disagreement stops the notebook.

In [ ]:
from src.quantum import VQCConfig, VQCLayer, circuit_resources, benchmark_simulator
vdef = {**qcfg["head_defaults"]["vqc"], "n_qubits": qcfg["head_defaults"]["k"]}
print(circuit_resources(VQCConfig(**vdef)))
crosscheck = {"enabled": qcfg["pennylane_crosscheck"]["enabled"]}
try:
    import pennylane as qml
    lt = VQCLayer(VQCConfig(**{**vdef, "backend": "torch"}))
    lp = VQCLayer(VQCConfig(**{**vdef, "backend": "pennylane"})); lp.weights.data = lt.weights.data.clone()
    print(lp.draw()); (ctx.dir("analysis") / "vqc_circuit.txt").write_text(lp.draw(), encoding="utf-8")
    x = torch.randn(qcfg["pennylane_crosscheck"]["n_samples"], vdef["n_qubits"])
    diff = float((lt(x) - lp(x)).abs().max())
    crosscheck.update(pennylane=qml.__version__, max_abs_diff=diff)
    assert diff < 1e-4, f"backends disagree: {diff}"
    bench_rows = benchmark_simulator(vdef, (64, 256)) + benchmark_simulator({**vdef, "backend": "pennylane"}, (64,), repeats=2)
except ImportError:
    crosscheck.update(pennylane="not installed", note="cross-check skipped; torch backend unit-tested against PennyLane offline")
    bench_rows = benchmark_simulator(vdef, (64, 256))
print(crosscheck)
bench = pd.DataFrame(bench_rows); display(bench)
atomic_save_dataframe(bench, ctx.dir("analysis") / "simulator_benchmark.csv")
atomic_write_json(ctx.dir("analysis") / "vqc_crosscheck.json", crosscheck)

## 4. Predetermined head grid x seeds (primary comparison + bounded ablations)
| id | role |
|---|---|
| `linear_probe` | upper reference: Linear(d->6) on the same features |
| `bottleneck_identity` | control: same k-dim encoder + readout, no processing block |
| `bottleneck_mlp` | control: VQC replaced by Linear(k,k)+tanh (parameter-matched) |
| `vqc_q4_l2_cyclic` | quantum-classical hybrid (Paper-2 circuit) |
| ablations | no entanglement (D), depth 1/4 (E), qubits 2/8 (F) + k=8 classical control |

In [ ]:
import copy
def head_cfg(spec):
    d = copy.deepcopy(qcfg["head_defaults"])
    d["kind"] = spec["kind"]
    if "k" in spec: d["k"] = spec["k"]
    d["vqc"] = {**d["vqc"], **spec.get("vqc", {})}
    if spec["kind"] != "vqc": d.pop("vqc")
    return d
grid = list(qcfg["primary"]) + (list(qcfg["ablations"]) if qcfg["run_ablations"] else [])
print(f"{len(grid)} head configurations x {len(qcfg['seeds'])} seeds = {len(grid) * len(qcfg['seeds'])} runs")
head_results = []
for spec in grid:
    for seed in qcfg["seeds"]:
        exp_id = f"C_{spec['id']}_s{seed}"
        r = run_head_experiment(ctx, exp_id, feats, head_cfg(spec), qcfg["train"], qcfg["loss"], frames=frames,
                                manifest_info=man_info, source_fingerprint=source_fp, seed=seed,
                                batch_size=qcfg["batch_size"], threshold_method=qcfg["threshold_method"], n_boot=qcfg["n_boot"])
        head_results.append({"head": spec["id"], "seed": seed, "status": r["status"],
                             "val_macro_auc": r.get("val_metrics", {}).get("macro_roc_auc", np.nan),
                             "test_macro_auc": r.get("test_metrics", {}).get("macro_roc_auc", np.nan),
                             "test_macro_ap": r.get("test_metrics", {}).get("macro_ap", np.nan),
                             "params": (r.get("head_params") or {}).get("total"),
                             "block_params": (r.get("head_params") or {}).get("block"),
                             "train_s": r.get("seconds_total"), "vqc_sim_s": r.get("vqc_sim_seconds"),
                             "epochs": (r.get("fit") or {}).get("epochs_run"), "error": r.get("error")})
hr = pd.DataFrame(head_results)
atomic_save_dataframe(hr, ctx.dir("analysis") / "trackC_runs.csv")
display(hr)

## 5. Summary across seeds and paired comparisons
Seed-level summary (mean, SD, t-interval) and paired tests across seeds; plus a patient-level paired bootstrap on the held-out test set using the seed-averaged predictions of each head.

In [ ]:
from src.evaluation.statistics import seed_summary, paired_seed_test, paired_bootstrap, holm
from src.evaluation.predictions import load_predictions, align, arrays
summ = []
for h, g in hr[hr["status"] == "succeeded"].groupby("head", sort=False):
    sv, st = seed_summary(g["val_macro_auc"]), seed_summary(g["test_macro_auc"])
    summ.append({"head": h, "n_seeds": st["n"], "val_mean": sv["mean"], "val_sd": sv["sd"], "test_mean": st["mean"],
                 "test_sd": st["sd"], "test_ci_low": st["ci_low"], "test_ci_high": st["ci_high"],
                 "params": int(g["params"].iloc[0]) if g["params"].notna().any() else None,
                 "mean_train_s": g["train_s"].mean(), "mean_vqc_sim_s": g["vqc_sim_s"].mean()})
summary = pd.DataFrame(summ); display(summary.round(4))
atomic_save_dataframe(summary, ctx.dir("analysis") / "trackC_summary.csv")

def seed_avg_preds(head):
    dfs = [load_predictions(ctx.dir("experiments", f"C_{head}_s{s}") / "predictions_test.parquet", man_info["manifest_hash"])[0]
           for s in qcfg["seeds"] if (ctx.dir("experiments", f"C_{head}_s{s}") / "predictions_test.parquet").exists()]
    if not dfs: return None
    ids, al = align({i: d for i, d in enumerate(dfs)})
    out = al[0].copy()
    probs = np.mean([arrays(d)[0] for d in al.values()], axis=0)
    for j, l in enumerate(LABELS): out[f"prob_{l}"] = probs[:, j]
    return out

QUANTUM = "vqc_q4_l2_cyclic"
comps = []
qa = seed_avg_preds(QUANTUM)
for ctrl in ["bottleneck_mlp", "bottleneck_identity", "linear_probe"]:
    ca = seed_avg_preds(ctrl)
    a, b = hr[(hr["head"] == QUANTUM) & (hr["status"] == "succeeded")], hr[(hr["head"] == ctrl) & (hr["status"] == "succeeded")]
    m = a.merge(b, on="seed", suffixes=("_q", "_c"))
    st = paired_seed_test(m["test_macro_auc_q"], m["test_macro_auc_c"]) if len(m) else {}
    rec = {"comparison": f"{QUANTUM} - {ctrl}", **{f"seeds_{k}": v for k, v in st.items()}}
    if qa is not None and ca is not None:
        ids, al = align({"q": qa, "c": ca})
        pq, t = arrays(al["q"]); pc_, _ = arrays(al["c"])
        rec.update({f"boot_{k}": v for k, v in paired_bootstrap(t, pq, pc_, al["q"]["group_id"].to_numpy(), "macro_auc",
                                                                 n_boot=max(qcfg["n_boot"], 100), seed=0).items() if k != "method"})
    comps.append(rec)
cmp_df = pd.DataFrame(comps)
if "boot_p_value" in cmp_df: cmp_df["boot_p_holm"] = holm(cmp_df["boot_p_value"])
display(cmp_df.round(4))
atomic_save_dataframe(cmp_df, ctx.dir("analysis") / "trackC_paired.csv")

## 6. Result statement generated from the numbers above (no pre-written conclusion)

In [ ]:
def statement(row):
    d, lo, hi = row.get("boot_difference", np.nan), row.get("boot_ci_low", np.nan), row.get("boot_ci_high", np.nan)
    if not np.isfinite(d):
        return f"{row['comparison']}: not evaluable (missing runs)."
    if lo > 0:
        verdict = "the quantum head is better, and the 95% patient-level CI excludes 0"
    elif hi < 0:
        verdict = "the quantum head is WORSE, and the 95% patient-level CI excludes 0"
    else:
        verdict = "no statistically reliable difference (CI includes 0)"
    return f"{row['comparison']}: delta macro AUC = {d:+.4f} [{lo:+.4f}, {hi:+.4f}] -> {verdict}. MODE={MODE}."
for _, r in cmp_df.iterrows():
    print(statement(r))
print("Simulated, noise-free circuit on classical hardware; no claim about quantum hardware or speed-up is made.")

## 7. Trainability and cost diagnostics

In [ ]:
from src.models.hybrid_classifier import FeatureHead
from src.quantum import angle_saturation
diag = []
for spec in grid:
    if spec["kind"] == "vqc":
        d = ctx.dir("experiments", f"C_{spec['id']}_s{qcfg['seeds'][0]}")
        if (d / "history.csv").exists():
            h = pd.read_csv(d / "history.csv")
            diag.append({"head": spec["id"], "mean_grad_norm": h["grad_norm"].mean(), "last_grad_norm": h["grad_norm"].iloc[-1],
                         "epochs": len(h)})
display(pd.DataFrame(diag))
hd = head_cfg(qcfg["primary"][-1]); head = FeatureHead(in_dim=feats["train"]["features"].shape[1], **hd)
from src.training.checkpointing import load_checkpoint
ck = ctx.dir("experiments", f"C_{qcfg['primary'][-1]['id']}_s{qcfg['seeds'][0]}") / "best.pt"
if ck.exists():
    load_checkpoint(ck, head, expected_manifest_hash=man_info["manifest_hash"])
    print("Angle saturation on validation features:", angle_saturation(head, torch.as_tensor(feats["val"]["features"][:4096])))